# ODA-LAB extract SET-2

**Runtime → Run all.** Separate window from SET-1.

Sunset splits + Lacey + the 416.6 MB tar last.
CAP 500 MB. Dest `12_ODA-LAB-NOTEBOOKS/extracts/SET2-<stamp>/`.
Do not dump the tar into pane 08. Originals stay put.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess
print('=== SET-2 mount ===')
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive')
print('mounted', ROOT.exists())
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
print('gdown ok')


In [ ]:
import gdown, zipfile, tarfile, time
CAP = 500 * 1024 * 1024
PACKS = [
  [
    "SUNSET_20260910_FULLGO_textpack.zip",
    "1VDCA_HFcwM-akFAmpFqZASp9N4R_Sfxg",
    25671,
    "zip"
  ],
  [
    "SUNSET_20260910_EXTRACT.zip",
    "1tzJs4edsPQ2A2sZoVPv1my-XdR72YrfK",
    352081,
    "zip"
  ],
  [
    "SUNSET_20260910_artifacts_remainder.zip",
    "1WijYI_JPnWtdrW5hrrUvqm_1hRRMM9Xg",
    251367,
    "zip"
  ],
  [
    "SUNSET_20260910_artifacts_rendered.zip",
    "1qzgnWAPjrIJ9QvzZSChXETgP4XKihQmd",
    3457908,
    "zip"
  ],
  [
    "LACEY-GROWN-20260910.zip",
    "1jwkbPpM3wlEi6PIz1XlZXqEsD3h5o_Je",
    2579896,
    "zip"
  ],
  [
    "SUNSET_20260910_claim_runtime_visuals.zip",
    "1TbP1gLrJL-XIQ_RpzOxc8rESjYlEbUTf",
    31134959,
    "zip"
  ],
  [
    "SUNSET_20260910_image_pipeline_visuals.zip",
    "1qMGCPsQvA2S0jQt6d_vARUr1GmB0Ecew",
    43052192,
    "zip"
  ],
  [
    "SUNSET_20260910_pixels_images.zip",
    "1ookVe_nUn9ubLZlpoD5DDoqePZfGWzrX",
    86050423,
    "zip"
  ],
  [
    "SUNSET_20260910_PIXELS.zip",
    "1VWXh1Jq3HMJ5vw8Se_Zo0L8fPJCR7S5M",
    97837561,
    "zip"
  ],
  [
    "SUNSET_FULL_20260910_grok-five.tar.gz",
    "1q0yQh5Qoxp3Q-p2QmlpCCXe6dWQNzAO_",
    416600000,
    "tar.gz"
  ]
]
shelf = None
nwalk = 0
for dirpath, dirnames, filenames in os.walk(ROOT):
    nwalk += 1
    if '12_ODA-LAB-NOTEBOOKS' in dirnames:
        shelf = Path(dirpath) / '12_ODA-LAB-NOTEBOOKS'
        break
    if nwalk > 4000:
        break
if shelf is None:
    shelf = ROOT / '12_ODA-LAB-NOTEBOOKS_LOCAL'
    shelf.mkdir(exist_ok=True)
dest_root = shelf / 'extracts' / ('SET2-' + time.strftime('%Y%m%d-%H%M'))
dest_root.mkdir(parents=True, exist_ok=True)
print('dest', dest_root)

def find_name(name):
    n = 0
    for dirpath, dirnames, filenames in os.walk(ROOT):
        n += len(filenames)
        if name in filenames:
            return Path(dirpath) / name
        if n > 12000:
            break
    return None

lines = ['# ODA LAB extract SET-2', 'dest=' + str(dest_root), '']
for name, fid, claimed, kind in PACKS:
    print('\n===', kind, name, fid)
    src = find_name(name)
    if src is None:
        local = Path('/content') / name
        print('gdown', fid)
        try:
            gdown.download(id=fid, output=str(local), quiet=False)
            src = local if local.is_file() else None
        except Exception as e:
            print('gdown fail', e)
            src = None
    if src is None or not src.is_file():
        lines.append('- FAIL missing ' + name)
        continue
    sz = src.stat().st_size
    print('src', src, 'size', sz)
    if sz > CAP:
        lines.append('- SKIP cap %s %s' % (name, sz))
        continue
    out = dest_root / src.name.split('.')[0]
    if out.exists() and any(out.iterdir()):
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- EXISTS %s files=%s' % (out, cnt))
        print('exists', cnt)
        continue
    out.mkdir(parents=True, exist_ok=True)
    try:
        if kind == 'zip':
            with zipfile.ZipFile(src) as z:
                z.extractall(out)
                nlist = z.namelist()[:6]
        else:
            with tarfile.open(src, 'r:*') as t:
                t.extractall(out)
                nlist = t.getnames()[:6]
        cnt = sum(1 for _ in out.rglob('*') if _.is_file())
        lines.append('- OK %s bytes=%s files=%s first=%s' % (name, sz, cnt, nlist))
        print('OK files', cnt)
    except Exception as e:
        lines.append('- FAIL extract %s %s' % (name, e))
        print('FAIL', e)
receipt = dest_root / 'EXTRACT.RECEIPT.md'
receipt.write_text('\n'.join(str(x) for x in lines))
print('\nWROTE', receipt)
print('\n'.join(str(x) for x in lines))
